In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

assert ".venv" in sys.executable, "Wrong kernel! Switch to 'Python (f1-tyre-strategy)' (top right)."
PROJECT_ROOT = Path.cwd().parent
assert PROJECT_ROOT.name == "f1-tyre-strategy-simulator", "Notebook isn't in the notebooks/ folder!"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
TARGET = "DeltaToQuali"
print("scikit-learn:", sklearn.__version__)
print("✅ Kernel and paths OK")

scikit-learn: 1.9.1
✅ Kernel and paths OK


In [2]:
df = pd.read_parquet(PROCESSED_DIR / "laps_clean.parquet")
print(f"Loaded: {df.shape[0]:,} laps × {df.shape[1]} columns")

# Tyre age is essential, so laps without it are dropped (decided in Stage 3)
no_age = df["TyreAge"].isna()
print(f"Laps without tyre age (dropped): {no_age.sum()}")
df = df[~no_age].copy()

# scikit-learn wants plain decimal numbers, not pandas' nullable whole-number type
for col in ["TyreAge", "GreenFlagTyreLaps", "LapNo", "StintNo", "LapsRemaining", "RaceLaps"]:
    df[col] = df[col].astype(float)

train = df[df["Split"] == "train"].copy()
val = df[df["Split"] == "val"].copy()
test = df[df["Split"] == "test"].copy()   # set aside: not used until the end of Stage 4

for name, part in [("train (2022–23)", train), ("val (2024)", val), ("test (2025)", test)]:
    print(f"{name:<16} {len(part):>6,} laps | {part[['Year', 'Round']].drop_duplicates().shape[0]} races")
print("\n🔒 The test set is locked away until the final step.")

Loaded: 67,823 laps × 21 columns
Laps without tyre age (dropped): 70
train (2022–23)  28,913 laps | 33 races
val (2024)       19,478 laps | 20 races
test (2025)      19,362 laps | 20 races

🔒 The test set is locked away until the final step.


In [3]:
def evaluate(data, pred, name):
    """Score predictions of DeltaToQuali. Lower is better for every metric except none (Bias should be near 0)."""
    err = pred - data[TARGET]
    # Stint total-time error: add up the errors over each driver's stint (clean laps only).
    # The simulator adds lap times together, so small biases grow into seconds here.
    stint_err = err.groupby([data["Year"], data["Round"], data["Driver"], data["StintNo"]]).sum()
    return {
        "Model": name,
        "MAE": err.abs().mean(),                  # average miss per lap (s)
        "RMSE": np.sqrt((err ** 2).mean()),       # punishes big misses more (s)
        "Bias": err.mean(),                       # + = predicts too slow, − = too fast (s)
        "StintTotalMAE": stint_err.abs().mean(),  # average miss on a whole stint's total time (s)
    }

results = []
print("✅ evaluate() ready")

✅ evaluate() ready


In [4]:
# B0: one number for every lap — the median of all training laps
b0_value = train[TARGET].median()
pred_b0 = pd.Series(b0_value, index=val.index)

# B1: median per circuit + compound in training; fall back to compound, then to B0
b1_table = train.groupby(["Circuit", "Compound"])[TARGET].median().rename("B1")
compound_median = train.groupby("Compound")[TARGET].median()
pred_b1 = val.join(b1_table, on=["Circuit", "Compound"])["B1"]
pred_b1 = pred_b1.fillna(val["Compound"].map(compound_median)).fillna(b0_value)

results.append(evaluate(val, pred_b0, "B0 constant"))
results.append(evaluate(val, pred_b1, "B1 track + compound"))
print(pd.DataFrame(results).round(3).to_string(index=False))

# How much worse is B1 on circuits it never saw in training?
unseen = ~val["Circuit"].isin(train["Circuit"].unique())
for label, mask in [("seen circuits", ~unseen), ("unseen circuits", unseen)]:
    mae = (pred_b1[mask] - val.loc[mask, TARGET]).abs().mean()
    print(f"B1 MAE on {label}: {mae:.3f} s ({mask.sum():,} laps)")

b1_mae = results[1]["MAE"]
print(f"\n🎯 Success target from Stage 1: validation MAE ≤ {0.85 * b1_mae:.3f} s (15% better than B1)")

              Model   MAE  RMSE   Bias  StintTotalMAE
        B0 constant 1.383 1.772 -0.462         25.969
B1 track + compound 1.037 1.390 -0.311         18.737
B1 MAE on seen circuits: 0.954 s (16,438 laps)
B1 MAE on unseen circuits: 1.487 s (3,040 laps)

🎯 Success target from Stage 1: validation MAE ≤ 0.882 s (15% better than B1)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

CAT = ["Compound", "Circuit"]
BASE_NUM = ["LapsRemaining", "RaceProgress", "TrackTemp", "AirTemp"]


def make_model(kind, tyre_col):
    """Return (pipeline, input columns). tyre_col is 'TyreAge' or 'GreenFlagTyreLaps'."""
    num = [tyre_col] + BASE_NUM
    one_hot = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
        ("num", "passthrough", num),
    ])
    if kind == "Ridge (linear)":
        prep, est = one_hot, Ridge(alpha=1.0)
    elif kind == "Random forest":
        prep, est = one_hot, RandomForestRegressor(
            n_estimators=200, min_samples_leaf=20, max_features=0.5, n_jobs=-1, random_state=42)
    elif kind == "Gradient boosting":
        prep = ColumnTransformer([
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), CAT),
            ("num", "passthrough", num),
        ])
        est = HistGradientBoostingRegressor(
            categorical_features=[0, 1], max_iter=300, learning_rate=0.05, random_state=42)
    else:
        raise ValueError(kind)
    return Pipeline([("prep", prep), ("model", est)]), CAT + num

print("✅ make_model() ready")

✅ make_model() ready


In [6]:
fitted = {}
for kind in ["Ridge (linear)", "Random forest", "Gradient boosting"]:
    for tyre_col in ["TyreAge", "GreenFlagTyreLaps"]:
        model, cols = make_model(kind, tyre_col)
        t0 = time.time()
        model.fit(train[cols], train[TARGET])
        pred = pd.Series(model.predict(val[cols]), index=val.index)
        name = f"{kind} | {tyre_col}"
        row = evaluate(val, pred, name)
        row["FitSeconds"] = round(time.time() - t0, 1)
        results.append(row)
        fitted[name] = (model, cols)
        print(f"Done: {name} ({row['FitSeconds']} s)")

table = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
table["MAE vs B1"] = (table["MAE"] / b1_mae - 1).map("{:+.1%}".format)
print()
print(table.round(3).to_string(index=False))

Done: Ridge (linear) | TyreAge (0.1 s)
Done: Ridge (linear) | GreenFlagTyreLaps (0.0 s)
Done: Random forest | TyreAge (0.7 s)
Done: Random forest | GreenFlagTyreLaps (0.7 s)
Done: Gradient boosting | TyreAge (1.1 s)
Done: Gradient boosting | GreenFlagTyreLaps (0.8 s)

                                Model   MAE  RMSE   Bias  StintTotalMAE  FitSeconds MAE vs B1
             Ridge (linear) | TyreAge 0.838 1.165 -0.335         13.905         0.1    -19.2%
   Ridge (linear) | GreenFlagTyreLaps 0.855 1.186 -0.337         14.274         0.0    -17.5%
          Gradient boosting | TyreAge 0.856 1.205 -0.381         14.715         1.1    -17.4%
Gradient boosting | GreenFlagTyreLaps 0.864 1.206 -0.378         14.930         0.8    -16.7%
              Random forest | TyreAge 0.922 1.275 -0.434         16.004         0.7    -11.1%
    Random forest | GreenFlagTyreLaps 0.924 1.274 -0.426         16.030         0.7    -11.0%
                  B1 track + compound 1.037 1.390 -0.311         18.737  

In [7]:
model, cols = fitted["Ridge (linear) | TyreAge"]
names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(model.named_steps["model"].coef_, index=names)

print("Numeric effects (seconds of lap time per 1 unit):")
print(coefs[[n for n in names if n.startswith("num__")]].round(4).to_string())

print("\nCompound effects (seconds, relative to the average compound):")
print(coefs[[n for n in names if n.startswith("cat__Compound")]].round(3).to_string())

Numeric effects (seconds of lap time per 1 unit):
num__TyreAge          0.0447
num__LapsRemaining   -0.0493
num__RaceProgress    -6.1900
num__TrackTemp        0.0178
num__AirTemp          0.0669

Compound effects (seconds, relative to the average compound):
cat__Compound_HARD      0.052
cat__Compound_MEDIUM    0.049
cat__Compound_SOFT     -0.101


In [8]:
# Make the src/ folder importable, then import the class (the app will do the same)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from features import TyreTerms
from sklearn.compose import make_column_selector

def make_ridge(compound_slopes, squares, circuit_slopes, alpha=1.0):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
        ("num", "passthrough", make_column_selector(dtype_include="number")),  # every numeric column, incl. new ones
    ])
    return Pipeline([
        ("terms", TyreTerms(compound_slopes=compound_slopes, squares=squares, circuit_slopes=circuit_slopes)),
        ("prep", prep),
        ("model", Ridge(alpha=alpha)),
    ])

RIDGE_NUM = ["TyreAge", "LapsRemaining", "TrackTemp", "AirTemp"]   # RaceProgress removed (collinear)
RIDGE_COLS = CAT + RIDGE_NUM

# Quick check: the class adds the expected columns
demo = TyreTerms().fit(train[RIDGE_COLS]).transform(train[RIDGE_COLS].head(3))
print("Columns after TyreTerms:", demo.shape[1], "| first new ones:", list(demo.columns[6:12]))
print("✅ TyreTerms imported from src/features.py")

Columns after TyreTerms: 33 | first new ones: ['Age_SOFT', 'AgeSq_SOFT', 'Age_MEDIUM', 'AgeSq_MEDIUM', 'Age_HARD', 'AgeSq_HARD']
✅ TyreTerms imported from src/features.py


In [9]:
ridge_variants = {
    "Ridge v2 | no RaceProgress":            dict(compound_slopes=False, squares=False, circuit_slopes=False),
    "Ridge v3 | + compound wear rates":      dict(compound_slopes=True,  squares=False, circuit_slopes=False),
    "Ridge v4 | + wear speeds up (squared)": dict(compound_slopes=True,  squares=True,  circuit_slopes=False),
    "Ridge v5 | + circuit wear rates":       dict(compound_slopes=True,  squares=True,  circuit_slopes=True),
}
for name, kw in ridge_variants.items():
    model = make_ridge(**kw)
    model.fit(train[RIDGE_COLS], train[TARGET])
    pred = pd.Series(model.predict(val[RIDGE_COLS]), index=val.index)
    results.append(evaluate(val, pred, name))
    fitted[name] = (model, RIDGE_COLS)

print(pd.DataFrame(results[-4:]).assign(**{"MAE vs B1": lambda t: (t["MAE"] / b1_mae - 1).map("{:+.1%}".format)})
      .round(3).to_string(index=False))

# What v3 learned: a wear rate per compound (seconds per lap of tyre age)
m3 = fitted["Ridge v3 | + compound wear rates"][0]
c3 = pd.Series(m3.named_steps["model"].coef_, index=m3.named_steps["prep"].get_feature_names_out())
print("\nv3 wear rate per compound (s per lap of tyre age):")
for comp in ["SOFT", "MEDIUM", "HARD"]:
    print(f"  {comp:<7} {c3['num__TyreAge'] + c3[f'num__Age_{comp}']:+.4f}")
print(f"Fuel effect (s per lap of fuel on board): {c3['num__LapsRemaining']:+.4f}")

                                Model   MAE  RMSE   Bias  StintTotalMAE MAE vs B1
           Ridge v2 | no RaceProgress 0.855 1.140 -0.337         14.357    -17.5%
     Ridge v3 | + compound wear rates 0.859 1.144 -0.341         14.491    -17.2%
Ridge v4 | + wear speeds up (squared) 0.869 1.160 -0.374         14.969    -16.2%
      Ridge v5 | + circuit wear rates 0.819 1.108 -0.288         13.486    -21.0%

v3 wear rate per compound (s per lap of tyre age):
  SOFT    +0.0697
  MEDIUM  +0.0485
  HARD    +0.0337
Fuel effect (s per lap of fuel on board): +0.0510


In [10]:
# Ridge: alpha controls how strongly coefficients are pulled towards 0 (more = simpler model)
best_kw = dict(compound_slopes=True, squares=True, circuit_slopes=True)
print("Ridge v5 — alpha tuning on 2024:")
for alpha in [0.1, 1, 10, 100]:
    m = make_ridge(**best_kw, alpha=alpha).fit(train[RIDGE_COLS], train[TARGET])
    mae = (m.predict(val[RIDGE_COLS]) - val[TARGET]).abs().mean()
    print(f"  alpha={alpha:<5} MAE {mae:.3f}")

# Gradient boosting: small grid, with a physics rule — an older tyre is never faster (monotonic in TyreAge)
GB_NUM = ["TyreAge", "LapsRemaining", "RaceProgress", "TrackTemp", "AirTemp"]
GB_COLS = CAT + GB_NUM
gb_prep = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), CAT),
    ("num", "passthrough", GB_NUM),
])
monotonic = [0, 0, 1, 0, 0, 0, 0]   # order: Compound, Circuit, TyreAge(+1), LapsRemaining, RaceProgress, TrackTemp, AirTemp

gb_rows = []
for lr in [0.03, 0.1]:
    for leaves in [15, 31]:
        for min_leaf in [50, 200]:
            for l2 in [0.0, 1.0]:
                est = HistGradientBoostingRegressor(
                    categorical_features=[0, 1], monotonic_cst=monotonic, max_iter=400,
                    learning_rate=lr, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
                    l2_regularization=l2, random_state=42)
                m = Pipeline([("prep", gb_prep), ("model", est)]).fit(train[GB_COLS], train[TARGET])
                mae = (m.predict(val[GB_COLS]) - val[TARGET]).abs().mean()
                gb_rows.append(dict(learning_rate=lr, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
                                    l2=l2, MAE=round(mae, 3)))
gb_grid = pd.DataFrame(gb_rows).sort_values("MAE")
print("\nGradient boosting — best 5 of 16 settings on 2024:")
print(gb_grid.head(5).to_string(index=False))

Ridge v5 — alpha tuning on 2024:
  alpha=0.1   MAE 0.819
  alpha=1     MAE 0.819
  alpha=10    MAE 0.822
  alpha=100   MAE 0.853

Gradient boosting — best 5 of 16 settings on 2024:
 learning_rate  max_leaf_nodes  min_samples_leaf  l2   MAE
          0.03              15               200 1.0 0.837
          0.03              15               200 0.0 0.842
          0.03              31               200 1.0 0.849
          0.03              31               200 0.0 0.851
          0.03              15                50 0.0 0.853


In [11]:
# Best Ridge alpha and best GB settings from Cell 10 (edit the alpha if Cell 10 printed a different winner)
BEST_ALPHA = 10
ridge_best = make_ridge(**best_kw, alpha=BEST_ALPHA).fit(train[RIDGE_COLS], train[TARGET])
results.append(evaluate(val, pd.Series(ridge_best.predict(val[RIDGE_COLS]), index=val.index),
                        f"Ridge v5 tuned (alpha={BEST_ALPHA})"))
fitted["Ridge v5 tuned"] = (ridge_best, RIDGE_COLS)

g = gb_grid.iloc[0]
gb_best = Pipeline([("prep", gb_prep), ("model", HistGradientBoostingRegressor(
    categorical_features=[0, 1], monotonic_cst=monotonic, max_iter=400,
    learning_rate=g["learning_rate"], max_leaf_nodes=int(g["max_leaf_nodes"]),
    min_samples_leaf=int(g["min_samples_leaf"]), l2_regularization=g["l2"], random_state=42))])
gb_best.fit(train[GB_COLS], train[TARGET])
results.append(evaluate(val, pd.Series(gb_best.predict(val[GB_COLS]), index=val.index), "Gradient boosting tuned"))
fitted["Gradient boosting tuned"] = (gb_best, GB_COLS)

leaderboard = pd.DataFrame(results).drop(columns=["FitSeconds"], errors="ignore").sort_values("MAE")
leaderboard["MAE vs B1"] = (leaderboard["MAE"] / b1_mae - 1).map("{:+.1%}".format)
print(leaderboard.round(3).to_string(index=False))

                                Model   MAE  RMSE   Bias  StintTotalMAE MAE vs B1
      Ridge v5 | + circuit wear rates 0.819 1.108 -0.288         13.486    -21.0%
            Ridge v5 tuned (alpha=10) 0.822 1.112 -0.292         13.541    -20.7%
              Gradient boosting tuned 0.837 1.192 -0.388         14.425    -19.3%
             Ridge (linear) | TyreAge 0.838 1.165 -0.335         13.905    -19.2%
           Ridge v2 | no RaceProgress 0.855 1.140 -0.337         14.357    -17.5%
   Ridge (linear) | GreenFlagTyreLaps 0.855 1.186 -0.337         14.274    -17.5%
          Gradient boosting | TyreAge 0.856 1.205 -0.381         14.715    -17.4%
     Ridge v3 | + compound wear rates 0.859 1.144 -0.341         14.491    -17.2%
Gradient boosting | GreenFlagTyreLaps 0.864 1.206 -0.378         14.930    -16.7%
Ridge v4 | + wear speeds up (squared) 0.869 1.160 -0.374         14.969    -16.2%
              Random forest | TyreAge 0.922 1.275 -0.434         16.004    -11.1%
    Random fores